# **1. Introdução**

O presente notebook tem como objetivo formatar a base de dados enviada pelo parceiro de projeto (BTG Pactual), a qual servirá como fonte para o treinamento do modelo de recomendação. A finalidade do projeto é a criação de um sistema que realizará uma reclassificação automática baseada nas regras definidas pela CVM (como estabelecido na Resolução CVM 175 e outros normativos aplicáveis), comparará com o perfil cadastrado e recomendará ajustes em casos de divergência.

Por meio deste processo, visamos garantir que o dataset esteja balanceado, apresentando uma distribuição uniforme de registros entre os diferentes usuários e itens. Ao final desse notebook, um arquivo nomeado 'dataset.csv' será gerado com as alterações aplicadas.

# **2. Carregamento do Dataset**

Nessa seção é realizada a importação do dataset enviado pelo parceiro, em formato de arquivo csv. As bibliotecas necessárias para a análise dos dados e formatação do arquivo também são importadas nessa etapa.

Instalação de bibliotecas

In [ ]:
!pip install imbalanced-learn


Importação das bibliotecas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from imblearn.under_sampling import RandomUnderSampler

Importação do dataset enviado pelo parceiro

In [ ]:
df = pd.read_csv('../aad/dados_carteiras.csv', sep=',')

# **3. Descrição do Dataset**

Nessa seção serão obtidas as informações descritivas do dataset, como informações sobre o número total de registros e colunas e exibição do cabeçalho do dataset com estatísticas descritivas.

O método head() exibe as primeiras linhas do DataFrame. - Ajuda a ter uma ideia rápida do conteúdo do DataFrame.

In [ ]:
df.head()

O método info() é usado para mostrar um resumo conciso do DataFrame. - Inclui Nome das colunas, número de entradas não nulas, tipo de dados (dtype), uso de memória, etc.

In [ ]:
df.info()

O método .shape serve para retornar a dimensão do DataFrame, em formato de tupla com (n_linhas, n_colunas)

In [ ]:
df.shape

O método describe() serve para gerar estatísticas descritivas das colunas numéricas - Média, desvio padrão, mínimo, quartis e máximo.

In [ ]:
df.describe()

Convertendo o tipo das colunas 'Quantidade' e 'Financeiro' para float64, pois apresentam apenas registros numéricos

In [ ]:
df['Financeiro'] = pd.to_numeric(df['Financeiro'].str.replace(',', '.'), errors='coerce')

In [ ]:
df['Quantidade'] = pd.to_numeric(df['Quantidade'].str.replace(',', '.'), errors='coerce')

In [ ]:
print(df['Quantidade'].dtype)

In [ ]:
print(df['Quantidade'].dtype)

Separa as colunas numéricas das categóricas para auxiliar na análise

In [ ]:
categoricas = df.select_dtypes(include=['object']).columns
numericas = df.select_dtypes(include=['number']).columns

print("Colunas categóricas: ", list(categoricas))
print("\nColunas numéricas: ", list(numericas))

Obtém informações das colunas numéricas

In [ ]:
df[numericas].info()

Obtém estatísticas das colunas numéricas

In [ ]:
df[['Financeiro', 'Quantidade']].describe()

# **4. Análise do balanceamento do Dataset**

Nessa seção é realizada a análise da distribuição dos rankings e verificação de como os dados estão balanceados, e são geradas visualizações para mostrar a distribuição dos rankings.

Verifica se as colunas numéricas apresentam valores nulos

In [ ]:
numericas.isnull().sum()

Verifica se as colunas categóricas apresentam valores nulos

In [ ]:
categoricas.isnull().sum()

Identifica outliers nas colunas 'Financeiro' e 'Quantidade' com boxplot. Através do gráfico de boxplot é possível perceber de forma visual a existência de pontos 'fora da curva', que estão afastados do restante. Isso é um indicativo de valores que podem estar muito acima ou muito abaixo da média.

In [ ]:
# Boxplot para 'Financeiro'
df.boxplot(column='Financeiro', grid=False)
plt.title('Boxplot de Financeiro')
plt.show()

# Boxplot para 'Quantidade'
df.boxplot(column='Quantidade', grid=False)
plt.title('Boxplot de Quantidade')
plt.show()

Aplica o método IQR para identificar os outliers. O IQR (Interquartile Range) é a diferença entre o 3º quartil (Q3) e o 1º quartil (Q1), a partir dos quais são definidos os limites superior e inferior, respectivamente. Dados que ultrapassam
esses limites são considerados outliers.

Análise IQR para coluna 'Financeiro'

In [ ]:
# Identificando outliers para a coluna 'Financeiro'
Q1_financeiro = df['Financeiro'].quantile(0.25)
Q3_financeiro = df['Financeiro'].quantile(0.75)
IQR_financeiro = Q3_financeiro - Q1_financeiro

limite_inferior_financeiro = Q1_financeiro - 1.5 * IQR_financeiro
limite_superior_financeiro = Q3_financeiro + 1.5 * IQR_financeiro

# Filtrando apenas os valores da coluna 'Financeiro' que são outliers
outliers_financeiro = df.loc[(df['Financeiro'] < limite_inferior_financeiro) | (df['Financeiro'] > limite_superior_financeiro), 'Financeiro']
print("Outliers em 'Financeiro':")
print(outliers_financeiro)

Análise IQR para coluna 'Quantidade'

In [ ]:
# Identificando outliers para a coluna 'Quantidade'
Q1_quantidade = df['Quantidade'].quantile(0.25)
Q3_quantidade = df['Quantidade'].quantile(0.75)
IQR_quantidade = Q3_quantidade - Q1_quantidade

limite_inferior_quantidade = Q1_quantidade - 1.5 * IQR_quantidade
limite_superior_quantidade = Q3_quantidade + 1.5 * IQR_quantidade

# Filtrando apenas os valores da coluna 'Quantidade' que são outliers
outliers_quantidade = df.loc[(df['Quantidade'] < limite_inferior_quantidade) | (df['Quantidade'] > limite_superior_quantidade), 'Quantidade']
print("Outliers em 'Quantidade':")
print(outliers_quantidade)

Os outliers identificados nas colunas numéricas Financeiro e Quantidade refletem variações reais nos registros, que podem ser relevantes para a análise e interpretação do comportamento dos ativos. Removê-los poderia resultar na perda de informações importantes sobre eventos extremos ou padrões específicos. Por esses motivos, optamos por mantê-los nesse momento.

### **Obtendo a distribuição dos registros das classes**

#### Análise de coluna Nome_ativo

10 ativos com maior número de registros

In [ ]:
distribuicao_nome_ativo = df['Nome_ativo'].value_counts().head(10)
print(distribuicao_nome_ativo)

Obtém a porcentagem em relação ao total dos 10 ativos com maior número de registros

In [ ]:
distribuicao_percentual = df['Nome_ativo'].value_counts(normalize=True).head(10) * 100
print(distribuicao_percentual)

A distribuição dos 10 ativos com maior número de registros é visualizada através de um gráfico de barras. Na coluna X estão os nomes de cada ativo e na coluna Y suas respectivas frequências no DataFrame

In [ ]:
plt.figure(figsize=(10, 6))
distribuicao_nome_ativo.plot(kind='bar', color='skyblue')
plt.title('Distribuição dos Ativos (Top 10)', fontsize=14)
plt.xlabel('Nome do Ativo', fontsize=12)
plt.ylabel('Frequência', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

10 ativos com o menor número de registros

In [ ]:
# Obtendo os 10 ativos com o menor número de registros
top_10_menores = df['Nome_ativo'].value_counts().tail(10)
print("\n10 classes com menor número de registros:")
print(top_10_menores)

#### **Análise da coluna 'Tipo_ativo'**

10 tipos de ativo com maior número de registros

In [ ]:
distribuicao_tipo_ativo = df['Tipo_ativo'].value_counts().head(10)
print(distribuicao_tipo_ativo)

Obtém a porcentagem em relação ao total dos 10 tipos de ativos com maior número de registros

In [ ]:
distribuicao_percentual_tipo = df['Tipo_ativo'].value_counts(normalize=True).head(10) * 100
print(distribuicao_percentual_tipo)

A distribuição dos 10 tipos de ativos com maior número de registros é visualizada através de um gráfico de barras. Na coluna X estão os nomes de cada tipo de ativo e na coluna Y suas respectivas frequências no DataFrame

In [ ]:
plt.figure(figsize=(10, 6))
distribuicao_tipo_ativo.plot(kind='bar', color='skyblue')
plt.title('Distribuição dos Tipos de Ativos (Top 10)', fontsize=14)
plt.xlabel('Tipo do Ativo', fontsize=12)
plt.ylabel('Frequência', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

#### **Análise da coluna 'Perfil_carteira'**

Obtém a quantidade de registros de cada tipo de perfil

In [ ]:
distribuicao_perfil = df['Perfil_carteira'].value_counts()
print(distribuicao_perfil)

A porcentagem relativa de cada tipo de perfil presente no DataFrame é visualizada através de um gráfico de setores

In [ ]:
# Gerando o gráfico de setores
plt.figure(figsize=(5, 5))
distribuicao_perfil.plot.pie(autopct='%1.1f%%', startangle=90, colors=plt.cm.Paired.colors)
plt.title('Distribuição dos Perfis de Carteira', fontsize=14)
plt.ylabel('')  # Remove o rótulo do eixo Y
plt.show()

#### **Análise de ativos classificados por tipo**

Agrupa os nomes de ativos por seus respectivos tipos

In [ ]:
ativos_por_tipo = df.groupby('Tipo_ativo')['Nome_ativo'].apply(list)
print(ativos_por_tipo)

Obtém os grupos de tipos com maior número de ativos

In [ ]:
tipos_com_mais_ativos = df.groupby('Tipo_ativo')['Nome_ativo'].nunique().sort_values(ascending=False).head(10)
print(tipos_com_mais_ativos)

A distribuição dos 10 grupos de tipos de ativos com maior número de ativos é visualizada através de um gráfico de barras. Na coluna X estão os nomes de cada grupo de tipos de ativo e na coluna Y a quantidade de ativos relacionados

In [ ]:
plt.figure(figsize=(10, 6))
tipos_com_mais_ativos.plot(kind='bar', color='skyblue')
plt.title('Grupos de Tipos de ativos com maior número de ativos (Top 10)', fontsize=14)
plt.xlabel('Tipo do Ativo', fontsize=12)
plt.ylabel('Número de ativos', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

Relaciona a coluna Quantidade com Nome_ativo - Compreendemos essa relação como os ativos que mais geraram lucro (retorno positivo)

In [ ]:
quantidade_por_ativo = df.groupby('Nome_ativo')['Quantidade'].sum().sort_values(ascending=False).head(10)
print(quantidade_por_ativo)

A distribuição dos 10 ativos em maior quantidade é visualizada através de um gráfico de barras. Na coluna X estão as quantidades de cada ativo e na coluna Y seus respectivos nomes.

In [ ]:
quantidade_por_ativo.plot(kind='barh', color='skyblue', figsize=(10, 6))
plt.title('Top 10 Ativos em maior quantidade', fontsize=14)
plt.xlabel('Quantidade', fontsize=12)
plt.ylabel('Nome do Ativo', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

A seguinte relação é compreendida como os 10 ativos que mais geraram prejuízo (retorno negativo)


In [ ]:
quantidade_por_ativo = df.groupby('Nome_ativo')['Quantidade'].sum().sort_values(ascending=False).tail(10)
print(quantidade_por_ativo)

#### **Aplicando o balanceamento dos dados**

Nessa etapa da análise, o objetivo é garantir que o dataset esteja balanceado, apresentando uma distribuição uniforme de registros entre os diferentes itens. Para atingir esse resultado, foi aplicada a técnica de Undersampling. Esse método reduz a quantidade de dados da classe majoritária (a que tem mais exemplos) para equilibrar com a classe minoritária. Isso é feito removendo aleatoriamente amostras da classe dominante. 

Essa técnica foi aplicada exclusivamente na coluna 'Tipo_ativos', pois foi a que apresentou a maior discrepância entre seus dados a partir das análises efetuadas.

Em etapas anteriores, observa-se que os 4 maiores registros dos tipos de ativo são os que apresentam a maior diferença em relação ao restante das classes. Por esse motivo, o undersampling é aplicado sobre elas.

In [ ]:
# Identificar os 4 ativos com maior número de registros
top_4_ativos = df['Tipo_ativo'].value_counts().head(4).index
df_top_4 = df[df['Tipo_ativo'].isin(top_4_ativos)]

Aplica-se uma redução de 50% no volume de registros dos 4 ativos identificados

In [ ]:

# Separar as features (X) e o target (y) para os 4 ativos
X_top_4 = df_top_4.drop(columns=['Tipo_ativo'])  
y_top_4 = df_top_4['Tipo_ativo']

# Calcular a redução proporcional 
class_counts = y_top_4.value_counts()
desired_counts = (class_counts * 0.5).astype(int).to_dict()  # Reduzir para 50%

# Aplicar o undersampling
rus = RandomUnderSampler(sampling_strategy=desired_counts, random_state=42)
X_resampled, y_resampled = rus.fit_resample(X_top_4, y_top_4)

# Combinar os dados reduzidos em um novo DataFrame
df_resampled_top_4 = pd.concat([pd.DataFrame(X_resampled, columns=X_top_4.columns), pd.DataFrame(y_resampled, columns=['Tipo_ativo'])], axis=1)

Retorna o novo volume de registros

In [ ]:
df_resampled_top_4.value_counts('Tipo_ativo')

In [ ]:
# Filtrar os registros que não estão entre os 4 ativos mais frequentes
df_restante = df[~df['Tipo_ativo'].isin(top_4_ativos)]

# Combinar os dados reduzidos com os demais registros
df_final = pd.concat([df_resampled_top_4, df_restante], axis=0)

Verifica a nova distribuição do DataFrame, após juntar os 4 registros balanceados com o restante

In [ ]:
# Verificar a nova distribuição
print(df_final['Tipo_ativo'].value_counts())

Obtém a distribuição final dos 10 maiores registros de ativos

In [ ]:
distribuicao_tipo_ativo_final = df_final['Tipo_ativo'].value_counts().head(10)
print(distribuicao_tipo_ativo_final)

A nova distribuição dos 10 ativos com maior número de registros é visualizada através de um gráfico de barras. Na coluna X estão os nomes de cada ativo e na coluna Y suas respectivas frequências no DataFrame

In [ ]:

distribuicao_tipo_ativo_final.plot(kind='bar', color='skyblue')
plt.title('Distribuição dos Tipos de Ativos (Top 10)', fontsize=14)
plt.xlabel('Tipo do Ativo', fontsize=12)
plt.ylabel('Frequência', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

Por fim, exporta-se o um novo Dataset com as alterações aplicadas em formato de arquivo csv

In [ ]:
# df.to_csv('dataset.csv', index=False)

# **5. Conclusão**

## Resumo das descobertas

**1. Distribuição dos Dados:**

&emsp;Identificou-se uma distribuição desigual entre os registros de diferentes tipos de ativos (Tipo_ativo), com algumas classes apresentando um número significativamente maior de registros.

**2. Outliers:**

&emsp;Foram detectados outliers nas colunas numéricas Financeiro e Quantidade utilizando o método IQR. Apesar disso, optou-se por mantê-los, considerando que representam variações reais nos dados.

**3. Balanceamento:**

&emsp;Aplicou-se a técnica de undersampling nos 4 tipos de ativos com maior número de registros, reduzindo-os em 50% para equilibrar a distribuição.

&emsp;Após o balanceamento, a nova distribuição dos dados foi verificada e mostrou-se mais uniforme.

**4. Qualidade dos Dados:**

&emsp;As colunas categóricas e numéricas foram separadas para análise, e os tipos de dados foram ajustados, como a conversão de Financeiro e Quantidade para float64.

&emsp;Não foram encontrados valores nulos nas colunas categóricas e numéricas após a limpeza inicial.

**5. Relações entre Colunas:**

&emsp;Identificou-se os ativos com maior e menor quantidade de registros, bem como os tipos de ativos mais representativos.

&emsp;A relação entre Nome_ativo e Quantidade revelou os ativos com maior impacto em termos de volume.

## **Próximos passos**

1. Validação do Balanceamento:

- Avaliar o impacto do undersampling na representatividade dos dados e na performance de modelos futuros.

2. Tratamento de Outliers (se necessário):

- Considerar técnicas para tratar outliers, como normalização ou transformação logarítmica, dependendo do impacto observado.

3. Feature Engineering:

- Criar novas variáveis derivadas das colunas existentes para enriquecer o dataset e melhorar a capacidade preditiva.

4. Treinamento de Modelos:

- Utilizar o dataset balanceado para treinar modelos de recomendação e avaliar sua performance.

5. Monitoramento Contínuo:

- Implementar um processo de monitoramento para garantir que futuras atualizações do dataset mantenham a qualidade e a consistência dos dados.